# Carte d'accès à la mobilité — vue quartier

Rassemble sur une seule carte, autour d'un point choisi :

| Couche | Source | Objectif du sujet |
|---|---|---|
| Carreaux de 200 m colorés du plus froid au plus chaud selon le nombre d'options de mobilité à 5 min à pied | `gold.france_access_walk_5min_national_200m` | 1. Cartographier l'accès |
| Arrêts et stations, taille selon les habitants desservis | `gold.france_stations_arrets_5min_population_desservie_clean` | 2. Qui est concerné |
| Sites Citiz recommandés par le scénario budgétaire | `gold.citiz_33c_scenarios_budget` | 3. Où s'implanter |
| Réseau piéton et zone réellement atteignable en 5 min depuis le point | `silver.reseau_pieton_*` | explique *pourquoi* un carreau est couvert ou non |

**Éthique** : les carreaux de moins de 11 habitants n'affichent pas leur effectif (vie privée) ; les libellés restent neutres (« aucune option à 5 min » plutôt que « mal desservi »).

⚠️ `lh_mobility_citiz` doit être le lakehouse **par défaut**.


In [1]:
%pip install folium networkx pyproj --quiet

## 1. Paramètres
Change `LIEU`, `CENTRE_LAT` et `CENTRE_LON` pour un autre endroit (clic droit sur Google Maps pour copier les coordonnées).

In [2]:
SILVER, GOLD = "silver", "gold"

# Point de départ de la démonstration (par défaut : place Kléber, Strasbourg)
LIEU       = "Place Kléber, Strasbourg"
CENTRE_LAT = 48.5834
CENTRE_LON = 7.7456

RAYON_CARTE_M  = 1000   # taille de la zone affichée autour du point
MINUTES_MARCHE = 5
VITESSE_KMH    = 4.32   # 1,2 m/s, la même hypothèse que le gold de l'équipe
DISTANCE_MAX_M = VITESSE_KMH * 1000 / 60 * MINUTES_MARCHE   # 5 min à 1,2 m/s = 360 m

# Tables gold de l'équipe affichées sur la carte
T_CARREAUX = f"{GOLD}.france_access_walk_5min_national_200m"
T_ARRETS   = f"{GOLD}.france_stations_arrets_5min_population_desservie_clean"
T_SITES    = f"{GOLD}.citiz_33c_scenarios_budget"

DOSSIER_CARTES = "/lakehouse/default/Files/silver/osm/cartes"

## 2. Réseau piéton (silver) sur la zone

In [3]:
import math
from pyspark.sql import functions as F

# Rectangle autour du point (1 degré de latitude ≈ 111 km)
dlat = RAYON_CARTE_M / 111_320
dlon = RAYON_CARTE_M / (111_320 * math.cos(math.radians(CENTRE_LAT)))

def dans_zone(col_lat, col_lon):
    return (F.col(col_lat).between(CENTRE_LAT - dlat, CENTRE_LAT + dlat)
            & F.col(col_lon).between(CENTRE_LON - dlon, CENTRE_LON + dlon))

troncons = (spark.table(f"{SILVER}.reseau_pieton_troncons")
            .filter(dans_zone("depart_latitude", "depart_longitude")
                    | dans_zone("arrivee_latitude", "arrivee_longitude"))
            .select("id_point_depart", "id_point_arrivee", "depart_longitude", "depart_latitude",
                    "arrivee_longitude", "arrivee_latitude", "longueur_m", "type_voie", "nom_voie")
            .toPandas())

points = (spark.table(f"{SILVER}.reseau_pieton_points")
          .filter(dans_zone("latitude", "longitude"))
          .toPandas())

print(f"{len(troncons):,} tronçons et {len(points):,} points dans la zone")
if troncons.empty:
    raise ValueError("Aucune donnée dans cette zone : la région a-t-elle bien été traitée par le notebook silver ?")
if len(troncons) > 40_000:
    print("⚠️ Zone très dense : la carte risque d'être lente. Réduis RAYON_CARTE_M (ex. 600).")

## 3. Résultats gold de l'équipe sur la zone

In [4]:
import pandas as pd

def lire(table, lat, lon, colonnes):
    """Lit une table gold sur la zone ; renvoie un tableau vide (avec un avertissement) si elle est absente."""
    try:
        return spark.table(table).filter(dans_zone(lat, lon)).select(*colonnes).toPandas()
    except Exception as e:
        print(f"⚠️ {table} illisible ({type(e).__name__}) : couche ignorée")
        return pd.DataFrame(columns=colonnes)

carreaux = lire(T_CARREAUX, "p_lat", "p_lon",
                ["idcar_200m", "population", "is_imputed", "p_lat", "p_lon", "statut_evaluation",
                 "nb_options_walk_5min", "nb_modes_walk_5min", "couvert_walk_5min", "nb_arrets_gtfs",
                 "nb_stations_citiz_avec_vehicule", "nb_velos_disponibles", "nb_trottinettes_disponibles",
                 "nb_voitures_gbfs_disponibles"])

arrets = lire(T_ARRETS, "latitude", "longitude",
              ["unique_id", "categorie", "latitude", "longitude", "population_desservie_5min",
               "nom", "operateur_affiche", "statut_calcul"])

sites_brut = lire(T_SITES, "latitude", "longitude",
                  ["budget_eur", "rang", "candidate_id", "latitude", "longitude", "nom_commune",
                   "population_desservie_5min", "population_nouvellement_couverte_marginale",
                   "cout_1ere_annee_eur", "utilisateurs_mois_central", "controle_terrain"])

print(f"{len(carreaux):,} carreaux de 200 m, {len(arrets):,} arrêts et stations, "
      f"{sites_brut.candidate_id.nunique():,} sites Citiz recommandés dans la zone")

# Aucun site Citiz recommandé ici ? On indique les plus proches pour pouvoir déplacer le point.
if sites_brut.empty:
    try:
        la0, lo0 = math.radians(CENTRE_LAT), math.radians(CENTRE_LON)
        dist_km = 2 * 6371.0088 * F.asin(F.sqrt(
            F.pow(F.sin((F.radians("latitude") - la0) / 2), 2)
            + math.cos(la0) * F.cos(F.radians("latitude")) * F.pow(F.sin((F.radians("longitude") - lo0) / 2), 2)))
        proches = (spark.table(T_SITES).select("candidate_id", "nom_commune", "latitude", "longitude")
                   .dropDuplicates(["candidate_id"]).withColumn("distance_km", F.round(dist_km, 1))
                   .orderBy("distance_km").limit(3).toPandas())
        print("Sites Citiz recommandés les plus proches (pour déplacer CENTRE_LAT / CENTRE_LON) :")
        for r in proches.itertuples():
            print(f"   • {r.nom_commune} : {r.distance_km} km → CENTRE_LAT = {r.latitude:.5f}, CENTRE_LON = {r.longitude:.5f}")
    except Exception as e:
        print(f"(recherche des sites les plus proches impossible : {type(e).__name__})")

## 4. Zone atteignable en 5 minutes et indicateurs

In [5]:
import networkx as nx
import numpy as np

def distance_m(lat1, lon1, lat2, lon2):
    """Distance à vol d'oiseau en mètres (fonctionne sur des colonnes entières)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6_371_008.8 * np.arcsin(np.sqrt(a))

# 1) Le graphe : chaque tronçon relie deux points, avec sa longueur
G = nx.Graph()
G.add_weighted_edges_from(zip(troncons.id_point_depart, troncons.id_point_arrivee, troncons.longueur_m),
                          weight="longueur_m")

# 2) Le point du réseau le plus proche du lieu choisi
points["dist_vol_oiseau_m"] = distance_m(CENTRE_LAT, CENTRE_LON, points.latitude, points.longitude)
depart = points.loc[points.dist_vol_oiseau_m.idxmin()]
print(f"Départ : point {depart.id_point}, à {depart.dist_vol_oiseau_m:.0f} m du lieu choisi")

# 3) On « marche » le long des chemins jusqu'à DISTANCE_MAX_M (algorithme de Dijkstra, celui des GPS)
atteints = nx.single_source_dijkstra_path_length(G, depart.id_point, cutoff=DISTANCE_MAX_M, weight="longueur_m")
troncons["atteint"] = troncons.id_point_depart.isin(atteints) & troncons.id_point_arrivee.isin(atteints)

# 4) Comparaison avec le cercle « à vol d'oiseau »
dans_cercle = points[points.dist_vol_oiseau_m <= DISTANCE_MAX_M]
part = dans_cercle.id_point.isin(atteints).mean() * 100
km_atteints = troncons.loc[troncons.atteint, "longueur_m"].sum() / 1000
print(f"Réseau parcourable en {MINUTES_MARCHE} min : {km_atteints:.1f} km de chemins")
print(f"Points à moins de {DISTANCE_MAX_M:.0f} m à vol d'oiseau : {len(dans_cercle):,}, "
      f"dont {part:.0f} % réellement atteignables")

# 5) Indicateurs gold de la zone
for c in ("population", "nb_options_walk_5min"):
    carreaux[c] = pd.to_numeric(carreaux[c], errors="coerce")
evalues = carreaux[carreaux.statut_evaluation == "evalue"]
pop_zone = evalues.population.sum()
pop_couverte = evalues.loc[evalues.couvert_walk_5min == True, "population"].sum()
taux_couv = 100 * pop_couverte / pop_zone if pop_zone else float("nan")
print(f"Habitants de la zone : {pop_zone:,.0f}, dont {taux_couv:.0f} % avec au moins une option à 5 min à pied")

# Un site Citiz apparaît dans plusieurs scénarios : on garde le plus petit budget qui le retient
sites = (sites_brut.sort_values(["candidate_id", "budget_eur", "rang"])
                   .drop_duplicates("candidate_id").reset_index(drop=True))

## 5. Carte

In [6]:
import os, re, unicodedata, folium
from pyproj import Transformer

def fr(x, dec=0):
    """Nombre au format français : 1 942 ; 9,5"""
    return f"{x:,.{dec}f}".replace(",", " ").replace(".", ",")

# ---------- Carreaux INSEE : le code donne le coin bas-gauche dans la projection officielle ----------
_projections = {}
def carre_wgs84(idcar, lat, lon):
    m = re.match(r"CRS(\d+)RES(\d+)mN(\d+)E(\d+)", str(idcar))
    if m:
        epsg, cote, n, e = map(int, m.groups())
        t = _projections.setdefault(epsg, Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True))
        coins = [(e, n), (e + cote, n), (e + cote, n + cote), (e, n + cote), (e, n)]
        return [[round(x, 6), round(y, 6)] for x, y in (t.transform(*c) for c in coins)]
    # repli : carré approximatif de 200 m autour du centre
    d_la, d_lo = 100 / 111_320, 100 / (111_320 * math.cos(math.radians(lat)))
    return [[lon - d_lo, lat - d_la], [lon + d_lo, lat - d_la], [lon + d_lo, lat + d_la],
            [lon - d_lo, lat + d_la], [lon - d_lo, lat - d_la]]

# Du plus froid au plus chaud, comme demandé par le sujet
CLASSES = [(0, 0, "Aucune option", "#2c7bb6"), (1, 2, "1 à 2", "#abd9e9"), (3, 5, "3 à 5", "#ffffbf"),
           (6, 10, "6 à 10", "#fdae61"), (11, 20, "11 à 20", "#f46d43"), (21, 10**9, "Plus de 20", "#a50026")]
NON_EVALUE = "#bdbdbd"

def couleur_classe(nb, statut):
    if statut != "evalue" or pd.isna(nb):
        return NON_EVALUE
    return next(c for lo, hi, _, c in CLASSES if lo <= nb <= hi)

def habitants(pop, impute):
    """Petits effectifs masqués (protection de la vie privée dans les zones peu peuplées)."""
    if pd.isna(pop):
        return "inconnu"
    if pop < 11:
        return "moins de 11"
    return fr(pop) + (" (estimé par l'INSEE)" if str(impute) in ("1", "True") else "")

# ---------- Réseau piéton (silver) ----------
FAMILLES = {
    "Cheminements piétons": ("#1a9850", ["Chemin piéton", "Rue piétonne", "Escalier", "Zone de rencontre", "Sentier"]),
    "Rues":                 ("#4d4d4d", ["Rue résidentielle", "Voie de desserte", "Petite route", "Voie non qualifiée"]),
    "Routes":               ("#252525", ["Route locale", "Route secondaire", "Route principale", "Voie rapide ouverte aux piétons"]),
    "Chemins ruraux et pistes cyclables": ("#8c510a", ["Chemin agricole ou forestier", "Chemin équestre",
                                                        "Piste cyclable ouverte aux piétons"]),
}
famille_de = {t: f for f, (_, types) in FAMILLES.items() for t in types}
troncons["famille"] = troncons.type_voie.map(famille_de).fillna("Autres")
COUL_FAMILLE = {f: c for f, (c, _) in FAMILLES.items()} | {"Autres": "#999999"}
COUL_ATTEINT = "#7b3294"

def geojson_troncons(df):
    return {"type": "FeatureCollection", "features": [
        {"type": "Feature",
         "geometry": {"type": "LineString", "coordinates": [[round(r.depart_longitude, 6), round(r.depart_latitude, 6)],
                                                            [round(r.arrivee_longitude, 6), round(r.arrivee_latitude, 6)]]},
         "properties": {"type": r.type_voie, "nom": r.nom_voie or "—", "longueur": f"{r.longueur_m:.0f} m"}}
        for r in df.itertuples()]}

infobulle_troncon = lambda: folium.GeoJsonTooltip(fields=["type", "nom", "longueur"],
                                                  aliases=["Type de voie", "Nom", "Longueur"])

# ---------- Carte ----------
m = folium.Map(location=[CENTRE_LAT, CENTRE_LON], zoom_start=16, tiles=None)
folium.TileLayer(
    tiles=("https://data.geopf.fr/wmts?SERVICE=WMTS&REQUEST=GetTile&VERSION=1.0.0"
           "&LAYER=GEOGRAPHICALGRIDSYSTEMS.PLANIGNV2&STYLE=normal&FORMAT=image/png"
           "&TILEMATRIXSET=PM&TILEMATRIX={z}&TILEROW={y}&TILECOL={x}"),
    attr="Fond : Plan IGN v2 © IGN – Géoplateforme", name="Fond : Plan IGN",
    max_zoom=19, max_native_zoom=19, opacity=0.5).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Fond : Esri, HERE, Garmin, © contributeurs OpenStreetMap", name="Fond : gris clair (Esri)",
    max_zoom=19, max_native_zoom=16, show=False).add_to(m)

# Calque A (gold, objectif 1) : carreaux de 200 m colorés selon le nombre d'options à 5 min
if not carreaux.empty:
    fc = {"type": "FeatureCollection", "features": [
        {"type": "Feature",
         "geometry": {"type": "Polygon", "coordinates": [carre_wgs84(r.idcar_200m, r.p_lat, r.p_lon)]},
         "properties": {
             "couleur": couleur_classe(r.nb_options_walk_5min, r.statut_evaluation),
             "habitants": habitants(r.population, r.is_imputed),
             "options": fr(r.nb_options_walk_5min) if r.statut_evaluation == "evalue" else "non évalué",
             "detail": (f"{fr(r.nb_arrets_gtfs or 0)} arrêts · {fr(r.nb_stations_citiz_avec_vehicule or 0)} Citiz · "
                        f"{fr(r.nb_velos_disponibles or 0)} vélos · {fr(r.nb_trottinettes_disponibles or 0)} trottinettes"
                        if r.statut_evaluation == "evalue" else "—")}}
        for r in carreaux.itertuples()]}
    folium.GeoJson(fc, name=f"Accès à 5 min par carreau de 200 m ({fr(len(carreaux))})",
                   style_function=lambda f: {"fillColor": f["properties"]["couleur"], "fillOpacity": 0.45,
                                             "color": "#ffffff", "weight": 0.6},
                   tooltip=folium.GeoJsonTooltip(fields=["habitants", "options", "detail"],
                                                 aliases=["Habitants", "Options à 5 min à pied", "Dont"])).add_to(m)

# Calque B (silver) : réseau piéton, masqué par défaut pour alléger la lecture
for famille, df in troncons.groupby("famille"):
    folium.GeoJson(geojson_troncons(df), name=f"Réseau : {famille.lower()}", show=False,
                   tooltip=infobulle_troncon(),
                   style_function=lambda _, c=COUL_FAMILLE[famille]: {"color": c, "weight": 1.5, "opacity": 0.6}
                   ).add_to(m)

# Calque C : ce qui est atteignable en 5 minutes depuis le point choisi
if troncons.atteint.any():
    folium.GeoJson(geojson_troncons(troncons[troncons.atteint]),
                   name=f"Atteignable en {MINUTES_MARCHE} min depuis {LIEU.split(',')[0]}",
                   tooltip=infobulle_troncon(),
                   style_function=lambda _: {"color": COUL_ATTEINT, "weight": 5, "opacity": 0.85}).add_to(m)
folium.Circle([CENTRE_LAT, CENTRE_LON], radius=DISTANCE_MAX_M, color=COUL_ATTEINT, weight=2,
              dash_array="8 6", fill=False, name=f"Cercle de {DISTANCE_MAX_M:.0f} m à vol d'oiseau").add_to(m)

# Calque D (gold, objectif 2) : arrêts et stations, taille selon la population desservie
LIBELLES = {"arret_gtfs": "Arrêts bus / tram / train", "arret_transport": "Arrêts bus / tram / train",
            "station_citiz": "Stations Citiz", "citiz": "Stations Citiz", "station_velo": "Stations vélos",
            "velo": "Stations vélos", "station_gbfs": "Stations partagées (GBFS)",
            "station_autopartage": "Stations d'autopartage", "trottinette": "Trottinettes"}
PALETTE = ["#1b9e77", "#d95f02", "#e7298a", "#66a61e", "#e6ab02", "#a6761d", "#666666"]
if not arrets.empty:
    arrets["population_desservie_5min"] = pd.to_numeric(arrets.population_desservie_5min, errors="coerce").fillna(0)
    for i, (cat, df) in enumerate(sorted(arrets.groupby("categorie"), key=lambda x: -len(x[1]))):
        libelle = LIBELLES.get(cat, str(cat).replace("_", " ").capitalize())
        coul = PALETTE[i % len(PALETTE)]
        groupe = folium.FeatureGroup(name=f"{libelle} ({fr(len(df))})")
        for r in df.itertuples():
            rayon = 4 + min(10, math.sqrt(r.population_desservie_5min) / 6)
            folium.CircleMarker(
                [r.latitude, r.longitude], radius=rayon, color="#ffffff", weight=1,
                fill=True, fill_color=coul, fill_opacity=0.95,
                tooltip=(f"<b>{r.nom or 'Sans nom'}</b><br>{libelle}"
                         f"{'<br>' + str(r.operateur_affiche) if r.operateur_affiche else ''}"
                         f"<br>Habitants à 5 min à pied : <b>{fr(r.population_desservie_5min)}</b>")).add_to(groupe)
        groupe.add_to(m)

# Calque E (gold, objectif 3) : sites Citiz recommandés
if not sites.empty:
    groupe = folium.FeatureGroup(name=f"Sites Citiz recommandés ({fr(len(sites))})")
    for r in sites.itertuples():
        folium.Marker(
            [r.latitude, r.longitude], icon=folium.Icon(color="darkred", icon="car", prefix="fa"),
            tooltip=(f"<b>Site Citiz recommandé</b> – {r.nom_commune or ''}"
                     f"<br>Retenu dès un budget de {fr(r.budget_eur)} € (rang {r.rang})"
                     f"<br>Habitants à 5 min : {fr(r.population_desservie_5min)}"
                     f" (dont {fr(r.population_nouvellement_couverte_marginale)} sans option aujourd'hui)"
                     f"<br>Coût 1re année : {fr(r.cout_1ere_annee_eur)} € · Usagers/mois estimés : {fr(r.utilisateurs_mois_central)}"
                     f"<br><i>{r.controle_terrain or ''}</i>")).add_to(groupe)
    groupe.add_to(m)

folium.Marker([depart.latitude, depart.longitude], tooltip=f"Point de départ : {LIEU}",
              icon=folium.Icon(color="purple", icon="user", prefix="fa")).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# ---------- Encadré explicatif ----------
case = lambda c, bord="": (f'<span style="display:inline-block;width:14px;height:14px;background:{c};{bord}'
                           f'margin-right:6px;vertical-align:middle;border-radius:2px"></span>')
legende = "".join(f"<div>{case(c)}{lib}</div>" for _, _, lib, c in CLASSES) + f"<div>{case(NON_EVALUE)}Non évalué</div>"
ligne_couv = (f"<b>{fr(pop_zone)}</b> habitants dans la zone, dont <b>{fr(taux_couv)} %</b> "
              f"avec au moins une option à {MINUTES_MARCHE} min à pied<br>" if pop_zone else "")
encadre = f"""
<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 14px;
            border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);font:12.5px/1.45 sans-serif;max-width:310px">
  <b>Accès à la mobilité à {MINUTES_MARCHE} min à pied</b><br>
  <span style="color:#666">autour de : {LIEU}</span>
  <div style="margin:6px 0">{ligne_couv}
    <span style="color:{COUL_ATTEINT}"><b>━━</b></span> atteignable depuis le point en suivant les rues
    ({fr(km_atteints, 1)} km)<br>
    <span style="color:{COUL_ATTEINT}"><b>┅┅</b></span> cercle de {DISTANCE_MAX_M:.0f} m :
    seuls <b>{fr(part)} %</b> des points sont réellement atteignables</div>
  <b>Options de mobilité à {MINUTES_MARCHE} min, par carreau de 200 m</b>
  {legende}
  <div style="margin-top:4px">● arrêts et stations (taille = habitants desservis) · 🚗 sites Citiz recommandés</div>
  <div style="color:#777;margin-top:6px">Population : INSEE Filosofi, carreaux 200 m (effectifs &lt; 11 masqués).
  Réseau : © contributeurs OpenStreetMap (ODbL). Transports : GTFS / GBFS (transport.data.gouv.fr).</div>
</div>"""
m.get_root().html.add_child(folium.Element(encadre))

# ---------- Sauvegarde ----------
os.makedirs(DOSSIER_CARTES, exist_ok=True)
lieu_court = unicodedata.normalize("NFKD", LIEU.split(",")[0]).encode("ascii", "ignore").decode().lower()
nom_fichier = f"{DOSSIER_CARTES}/carte_acces_mobilite_{lieu_court.replace(' ', '_')}.html"
m.save(nom_fichier)
print("Carte enregistrée :", nom_fichier.replace("/lakehouse/default/", ""))

m